# PhytoFactor-HPCTR— Cell-wise Ablation Study Notebook

This notebook is **only for ablation experiments**. It does **not** retrain the full proposed model.

The experimental protocol remains matched to the main `plantseg01.ipynb` experiment:

- PlantSegV2 official train/validation/test folders.
- ConvNeXt-Tiny backbone and the same image size/training hyperparameters.
- Validation-only checkpoint selection using the same composite score.
- EMA weights for validation and the final test evaluation.
- **Primary single-view test inference only** for fair ablation comparison.
- The test set is never used for model selection.


### Default reproducibility settings

All dedicated cells use the same `DEFAULT_SEED = 42`, `RESUME = True`, and `FORCE_RETRAIN = False`. These are defined once in the runner cell and normally should not be changed during the first ablation pass.


In [2]:
# 1. Imports, reproducibility, and global configuration
import os, re, gc, math, time, json, random, warnings, copy
from pathlib import Path
from dataclasses import dataclass, asdict
from collections import defaultdict

import numpy as np
import pandas as pd
from PIL import Image
from IPython.display import display

from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    mean_absolute_error, mean_squared_error, r2_score
)
from scipy.stats import pearsonr, spearmanr
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.models import convnext_tiny, ConvNeXt_Tiny_Weights
from torchvision.transforms import functional as TF
from torchvision.transforms import InterpolationMode

warnings.filterwarnings('ignore')
torch.backends.cudnn.benchmark = True
try:
    torch.set_float32_matmul_precision('high')
except Exception:
    pass

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

@dataclass
class CFG:
    # Same primary protocol as the main notebook
    seed: int = 42
    image_size: int = 512
    epochs: int = 34
    batch_size: int = 2
    grad_accum: int = 2
    num_workers: int = 0
    prefetch_factor: int = 2
    pretrained: bool = True

    backbone_lr: float = 3e-5
    head_lr: float = 3e-4
    layer_decay: float = 0.80
    weight_decay: float = 0.05
    warmup_epochs: int = 2
    min_lr_ratio: float = 0.04
    grad_clip: float = 1.0

    fpn_dim: int = 128
    embed_dim: int = 256
    dropout: float = 0.20

    ema_decay: float = 0.9995
    ema_warmup_tau: float = 2000.0
    label_smoothing: float = 0.03
    lovasz_downsample: int = 256

    patience: int = 9
    progress_every: int = 20

    # Main loss weights
    w_combo: float = 1.00
    w_host: float = 0.20
    w_disease: float = 0.20
    w_taxonomy: float = 0.10
    w_prototype: float = 0.30
    w_seg: float = 1.55
    w_factorized: float = 0.45
    w_severity: float = 0.50
    w_consistency: float = 0.08

    # Same coverage-preserving augmentation
    hflip_p: float = 0.50
    vflip_p: float = 0.10
    rot90_p: float = 0.25
    color_jitter_p: float = 0.70

    # Ablation output root
    output_root: str = 'ablation'

cfg = CFG()
DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
AMP_ENABLED = DEVICE.type == 'cuda'
Path(cfg.output_root).mkdir(parents=True, exist_ok=True)
seed_everything(cfg.seed)

print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
print('Device:', DEVICE)
print('Ablation root:', cfg.output_root)
print('Effective batch size:', cfg.batch_size * cfg.grad_accum)


PyTorch: 2.5.1
CUDA: True
GPU: NVIDIA RTX 2000 Ada Generation Laptop GPU
Device: cuda:0
Ablation root: ablation
Effective batch size: 4


In [3]:

# 2. Official PlantSeg 115-combination taxonomy
COMBO_NAMES = [
'apple black rot','apple mosaic virus','apple rust','apple scab',
'banana anthracnose','banana black leaf streak','banana bunchy top','banana cigar end rot','banana cordana leaf spot','banana panama disease',
'basil downy mildew','bean halo blight','bean mosaic virus','bean rust',
'bell pepper bacterial spot','bell pepper blossom end rot','bell pepper frogeye leaf spot','bell pepper powdery mildew',
'blueberry anthracnose','blueberry botrytis blight','blueberry mummy berry','blueberry rust','blueberry scorch',
'broccoli alternaria leaf spot','broccoli downy mildew','broccoli ring spot','cabbage alternaria leaf spot','cabbage black rot','cabbage downy mildew',
'carrot alternaria leaf blight','carrot cavity spot','carrot cercospora leaf blight','cauliflower alternaria leaf spot','cauliflower bacterial soft rot',
'celery anthracnose','celery early blight','cherry leaf spot','cherry powdery mildew','citrus canker','citrus greening disease',
'coffee berry blotch','coffee black rot','coffee brown eye spot','coffee leaf rust','corn gray leaf spot','corn northern leaf blight','corn rust','corn smut',
'cucumber angular leaf spot','cucumber bacterial wilt','cucumber powdery mildew','eggplant cercospora leaf spot','eggplant phomopsis fruit rot','eggplant phytophthora blight',
'garlic leaf blight','garlic rust','ginger leaf spot','ginger sheath blight','grape black rot','grape downy mildew','grape leaf spot','grapevine leafroll disease',
'lettuce downy mildew','lettuce mosaic virus','maple tar spot','peach anthracnose','peach brown rot','peach leaf curl','peach rust','peach scab',
'plum bacterial spot','plum brown rot','plum pocket disease','plum pox virus','plum rust','potato early blight','potato late blight',
'raspberry fire blight','raspberry gray mold','raspberry leaf spot','raspberry yellow rust','rice blast','rice sheath blight',
'soybean bacterial blight','soybean brown spot','soybean downy mildew','soybean frog eye leaf spot','soybean mosaic','soybean rust','squash powdery mildew',
'strawberry anthracnose','strawberry leaf scorch','tobacco blue mold','tobacco brown spot','tobacco frogeye leaf spot','tobacco mosaic virus',
'tomato bacterial leaf spot','tomato early blight','tomato late blight','tomato leaf mold','tomato mosaic virus','tomato septoria leaf spot','tomato yellow leaf curl virus',
'wheat bacterial leaf streak (black chaff)','wheat head scab','wheat leaf rust','wheat loose smut','wheat powdery mildew','wheat septoria blotch','wheat stem rust','wheat stripe rust',
'zucchini bacterial wilt','zucchini downy mildew','zucchini powdery mildew','zucchini yellow mosaic virus']
assert len(COMBO_NAMES) == 115

def split_combo_taxonomy(combo):
    if combo.startswith('bell pepper '):
        host='bell pepper'; disease=combo[len('bell pepper '):]
    elif combo.startswith('grapevine '):
        host='grape'; disease=combo[len('grapevine '):]
    else:
        host=combo.split()[0]; disease=combo[len(host)+1:]
    disease=disease.replace('frog eye','frogeye')
    return host,disease

combo_tax=[split_combo_taxonomy(x) for x in COMBO_NAMES]
HOST_NAMES=sorted({h for h,d in combo_tax})
DISEASE_NAMES=sorted({d for h,d in combo_tax})
assert len(HOST_NAMES)==34
assert len(DISEASE_NAMES)==69
host_to_idx={x:i for i,x in enumerate(HOST_NAMES)}
disease_to_idx={x:i for i,x in enumerate(DISEASE_NAMES)}
COMBO_TO_HOST=torch.tensor([host_to_idx[h] for h,d in combo_tax],dtype=torch.long)
COMBO_TO_DISEASE=torch.tensor([disease_to_idx[d] for h,d in combo_tax],dtype=torch.long)
print(len(COMBO_NAMES), len(HOST_NAMES), len(DISEASE_NAMES))


115 34 69


In [4]:
# 3. PlantSegV2 path, image-mask pairing, and binary lesion decoding
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
DATA_ROOT = Path('plantsegv2')
if not DATA_ROOT.exists():
    raise FileNotFoundError(f'Dataset path not found: {DATA_ROOT}')

def norm_key(s):
    return re.sub(r'[^a-z0-9]+', '_', str(s).lower()).strip('_')

COMBO_SLUGS = sorted(
    [(i, norm_key(n)) for i, n in enumerate(COMBO_NAMES)],
    key=lambda x: len(x[1]), reverse=True
)

def infer_combo_from_filename(filename):
    stem = norm_key(Path(filename).stem)
    for idx, slug in COMBO_SLUGS:
        if stem == slug or stem.startswith(slug + '_'):
            return idx
    return None

def split_dir(parent, split):
    p = DATA_ROOT / parent / split
    if not p.exists():
        raise FileNotFoundError(f'Folder not found: {p}')
    return p

def infer_combo_from_mask_file(mask_path):
    a = np.array(Image.open(mask_path))
    vals = np.unique(a)
    nz = [int(v) for v in vals if 1 <= int(v) <= 115]
    if len(nz) == 1 and 0 in vals:
        return nz[0] - 1
    cand = [int(v) for v in vals if 0 <= int(v) <= 114]
    if len(cand) == 1 and 255 in vals:
        return cand[0]
    return None

def build_split_index(split):
    imgs = [p for p in split_dir('images', split).rglob('*')
            if p.is_file() and p.suffix.lower() in IMAGE_EXTS]
    masks = [p for p in split_dir('annotations', split).rglob('*.png')]
    mask_by_stem = {p.stem: p for p in masks}
    rows, missing, unresolved = [], [], []
    for ip in sorted(imgs):
        mp = mask_by_stem.get(ip.stem)
        if mp is None:
            missing.append(ip.name)
            continue
        combo = infer_combo_from_filename(ip.name)
        if combo is None:
            combo = infer_combo_from_mask_file(mp)
        if combo is None:
            unresolved.append(ip.name)
            continue
        h = int(COMBO_TO_HOST[combo])
        d = int(COMBO_TO_DISEASE[combo])
        rows.append(dict(
            split=split, filename=ip.name, image_path=str(ip), mask_path=str(mp),
            combo=combo, host=h, disease=d,
            combo_name=COMBO_NAMES[combo], host_name=HOST_NAMES[h], disease_name=DISEASE_NAMES[d]
        ))
    print(f'{split}: paired={len(rows)} missing_mask={len(missing)} unresolved={len(unresolved)}')
    return pd.DataFrame(rows)

train_df = build_split_index('train')
val_df = build_split_index('val')
test_df = build_split_index('test')
assert len(train_df) and len(val_df) and len(test_df)
assert train_df.combo.nunique() > 100, 'Too few classes recovered.'

# Split leakage sanity check
train_files, val_files, test_files = set(train_df.filename), set(val_df.filename), set(test_df.filename)
assert not (train_files & val_files), 'Filename overlap between train and validation.'
assert not (train_files & test_files), 'Filename overlap between train and test.'
assert not (val_files & test_files), 'Filename overlap between validation and test.'

def mask_to_binary(arr, combo_idx):
    arr = np.asarray(arr)
    vals = set(np.unique(arr).astype(int).tolist())
    if 0 in vals and (combo_idx + 1) in vals:
        return (arr == (combo_idx + 1)).astype(np.uint8)
    if 255 in vals and combo_idx in vals:
        return (arr == combo_idx).astype(np.uint8)
    if vals.issubset({0, 255}):
        return (arr == 255).astype(np.uint8)
    pos = (arr != 0)
    if 255 in vals:
        pos = (arr != 0) & (arr != 255)
    return pos.astype(np.uint8)

display(pd.DataFrame({
    'split': ['train', 'val', 'test'],
    'images': [len(train_df), len(val_df), len(test_df)],
    'classes': [train_df.combo.nunique(), val_df.combo.nunique(), test_df.combo.nunique()]
}))


train: paired=7916 missing_mask=0 unresolved=0
val: paired=1247 missing_mask=0 unresolved=0
test: paired=2295 missing_mask=0 unresolved=0


,split,images,classes
0,train,7916,115
1,val,1247,115
2,test,2295,115


In [5]:
# 4. Dataset, augmentation, and reproducible per-run DataLoaders
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

class PlantSegMTLDataset(Dataset):
    def __init__(self, df, train=False):
        self.df = df.reset_index(drop=True)
        self.train = train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        r = self.df.iloc[idx]
        image = Image.open(r.image_path).convert('RGB')
        raw = np.array(Image.open(r.mask_path))
        binary = mask_to_binary(raw, int(r.combo))
        severity = float(binary.mean())
        mask = Image.fromarray((binary * 255).astype(np.uint8), mode='L')

        image = TF.resize(
            image, [cfg.image_size, cfg.image_size],
            interpolation=InterpolationMode.BILINEAR, antialias=True
        )
        mask = TF.resize(mask, [cfg.image_size, cfg.image_size], interpolation=InterpolationMode.NEAREST)

        if self.train:
            if random.random() < cfg.hflip_p:
                image, mask = TF.hflip(image), TF.hflip(mask)
            if random.random() < cfg.vflip_p:
                image, mask = TF.vflip(image), TF.vflip(mask)
            if random.random() < cfg.rot90_p:
                angle = 90 * random.choice([1, 2, 3])
                image = TF.rotate(image, angle, interpolation=InterpolationMode.BILINEAR)
                mask = TF.rotate(mask, angle, interpolation=InterpolationMode.NEAREST)
            if random.random() < cfg.color_jitter_p:
                image = TF.adjust_brightness(image, random.uniform(.87, 1.13))
                image = TF.adjust_contrast(image, random.uniform(.87, 1.13))
                image = TF.adjust_saturation(image, random.uniform(.87, 1.13))
                image = TF.adjust_hue(image, random.uniform(-.025, .025))

        image = TF.normalize(TF.to_tensor(image), IMAGENET_MEAN, IMAGENET_STD)
        mask = (TF.pil_to_tensor(mask).float() / 255.0 > 0.5).float()
        return {
            'image': image,
            'mask': mask,
            'combo': torch.tensor(int(r.combo), dtype=torch.long),
            'host': torch.tensor(int(r.host), dtype=torch.long),
            'disease': torch.tensor(int(r.disease), dtype=torch.long),
            'severity': torch.tensor(severity, dtype=torch.float32),
            'filename': r.filename
        }

train_ds = PlantSegMTLDataset(train_df, True)
val_ds = PlantSegMTLDataset(val_df, False)
test_ds = PlantSegMTLDataset(test_df, False)

def make_loaders(run_seed):
    # Rebuild loaders for every independent ablation so runs do not inherit worker/shuffle state.
    seed_everything(run_seed)
    generator = torch.Generator()
    generator.manual_seed(run_seed)

    loader_args = dict(
        num_workers=cfg.num_workers,
        pin_memory=AMP_ENABLED,
        persistent_workers=cfg.num_workers > 0
    )
    if cfg.num_workers > 0:
        loader_args['prefetch_factor'] = cfg.prefetch_factor

    train_loader = DataLoader(
        train_ds, batch_size=cfg.batch_size, shuffle=True, drop_last=True,
        generator=generator, **loader_args
    )
    val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False, **loader_args)
    test_loader = DataLoader(test_ds, batch_size=cfg.batch_size, shuffle=False, **loader_args)
    return train_loader, val_loader, test_loader

print('Dataset objects ready.')


Dataset objects ready.


In [6]:
# 5. Long-tail class weights (same construction as the main notebook)
def effective_num_weights(labels, n_classes, beta=.999, clip=(.35, 5.0)):
    counts = np.bincount(np.asarray(labels, dtype=int), minlength=n_classes).astype(np.float64)
    cs = np.maximum(counts, 1.0)
    w = (1 - beta) / (1 - np.power(beta, cs))
    w = w / w.mean()
    w = np.clip(w, *clip)
    return torch.tensor(w, dtype=torch.float32), counts

combo_w_cpu, combo_counts = effective_num_weights(train_df.combo.values, 115)
host_w_cpu, host_counts = effective_num_weights(train_df.host.values, 34)
disease_w_cpu, disease_counts = effective_num_weights(train_df.disease.values, 69)

print('Combo count min/median/max:', int(combo_counts.min()), float(np.median(combo_counts)), int(combo_counts.max()))


Combo count min/median/max: 5 53.0 271


## Ablation design

### Core architecture ablations

- **A01 — No HPCTR routing:** replace task-specific routing with shared projected features.
- **A02 — No cross-route transfer:** keep HPCTR routing but remove the directed identity→lesion and lesion→severity transfers.
- **A03 — No prototype mechanism:** remove prototype classification and prototype-conditioned routing.
- **A04 — No taxonomy prior:** remove the bounded host/disease taxonomy prior from 115-class logits while retaining auxiliary host/disease learning.
- **A05 — No identity refinement:** remove the routed identity refinement added to combination logits.
- **A06 — No uncertainty-gated severity fusion:** replace the learned uncertainty-aware gate with a fixed 50/50 mask/direct fusion and suppress the uncertainty feature.

### Objective/support ablations

- **A07 — No factorized coupling loss:** remove the lesion-weighted classification/segmentation coupling term.
- **A08 — No taxonomy consistency loss:** remove JS consistency between combination marginals and host/disease heads.
- **A09 — Basic segmentation loss:** use only Dice + Focal instead of Dice + Focal + Lovász + boundary Dice.
- **A10 — No class balancing:** use unweighted cross-entropy for combination/host/disease/prototype classification.

In [7]:
# 6. Ablation registry — full model intentionally excluded
BASE_SPEC = dict(
    use_hpctr=True,
    use_cross_transfer=True,
    use_prototypes=True,
    use_taxonomy_prior=True,
    use_identity_refine=True,
    severity_mode='uncertainty_gate',   # alternative used here: 'fixed50'
    use_factorized_loss=True,
    use_taxonomy_consistency=True,
    seg_loss_mode='full',               # alternative: 'dice_focal'
    use_class_balancing=True,
)

def make_spec(group, change, **kwargs):
    d = dict(BASE_SPEC)
    d.update(kwargs)
    d['group'] = group
    d['change'] = change
    return d

ABLATIONS = {
    'A01_no_hpctr_routing': make_spec(
        'core_architecture',
        'HPCTR router removed; all tasks receive the same projected feature at each scale.',
        use_hpctr=False
    ),
    'A02_no_cross_route_transfer': make_spec(
        'core_architecture',
        'HPCTR routing retained, but identity→lesion and lesion→severity transfers are removed.',
        use_cross_transfer=False
    ),
    'A03_no_prototype_mechanism': make_spec(
        'core_architecture',
        'Prototype classifier, prototype logit contribution, prototype loss, and prototype conditioning are removed.',
        use_prototypes=False
    ),
    'A04_no_taxonomy_prior': make_spec(
        'core_architecture',
        'Bounded host/disease taxonomy prior is removed from 115-class combination logits.',
        use_taxonomy_prior=False
    ),
    'A05_no_identity_refinement': make_spec(
        'core_architecture',
        'Final identity-route refinement of combination logits is removed.',
        use_identity_refine=False
    ),
    'A06_no_uncertainty_gate': make_spec(
        'core_architecture',
        'Learned uncertainty-aware severity gate is replaced by fixed 50/50 mask/direct fusion; uncertainty input is zeroed.',
        severity_mode='fixed50'
    ),
    'A07_no_factorized_coupling': make_spec(
        'objective_support',
        'Lesion-weighted factorized classification/segmentation coupling loss is removed.',
        use_factorized_loss=False
    ),
    'A08_no_taxonomy_consistency': make_spec(
        'objective_support',
        'JS consistency between host/disease heads and combination marginals is removed.',
        use_taxonomy_consistency=False
    ),
    'A09_basic_segmentation_loss': make_spec(
        'objective_support',
        'Segmentation objective is reduced to 0.5 Dice + 0.5 Focal; Lovasz and boundary terms are removed.',
        seg_loss_mode='dice_focal'
    ),
    'A10_no_class_balancing': make_spec(
        'objective_support',
        'Effective-number class weighting is removed from classification losses.',
        use_class_balancing=False
    ),
}

ablation_overview = pd.DataFrame([
    {'key': k, 'group': v['group'], 'change': v['change']} for k, v in ABLATIONS.items()
])
display(ablation_overview)


,key,group,change
0,A01_no_hpctr_routing,core_architecture,HPCTR router removed; all tasks receive the sa...
1,A02_no_cross_route_transfer,core_architecture,"HPCTR routing retained, but identity→lesion an..."
2,A03_no_prototype_mechanism,core_architecture,"Prototype classifier, prototype logit contribu..."
3,A04_no_taxonomy_prior,core_architecture,Bounded host/disease taxonomy prior is removed...
4,A05_no_identity_refinement,core_architecture,Final identity-route refinement of combination...
5,A06_no_uncertainty_gate,core_architecture,Learned uncertainty-aware severity gate is rep...
6,A07_no_factorized_coupling,objective_support,Lesion-weighted factorized classification/segm...
7,A08_no_taxonomy_consistency,objective_support,JS consistency between host/disease heads and ...
8,A09_basic_segmentation_loss,objective_support,Segmentation objective is reduced to 0.5 Dice ...
9,A10_no_class_balancing,objective_support,Effective-number class weighting is removed fr...


In [8]:
# 7. Ablatable PhytoFactor-HPCTR v2 model
class ConvGNAct(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        g = 8 if cout % 8 == 0 else 1
        self.block = nn.Sequential(
            nn.Conv2d(cin, cout, 3, padding=1, bias=False),
            nn.GroupNorm(g, cout),
            nn.GELU()
        )

    def forward(self, x):
        return self.block(x)

class SharedPassBlock(nn.Module):
    """A01: no HPCTR routing; retain matched normalization without task routing/transfer."""
    def __init__(self, channels):
        super().__init__()
        g = 8 if channels % 8 == 0 else 1
        self.lesion_norm = nn.GroupNorm(g, channels)
        self.severity_norm = nn.GroupNorm(g, channels)

    def forward(self, feature, soft_proto=None):
        b = feature.size(0)
        route_weights = feature.new_full((b, 3, feature.size(1), 1, 1), 1.0 / 3.0)
        return feature, self.lesion_norm(feature), self.severity_norm(feature), route_weights

class HPCTRBlock(nn.Module):
    def __init__(self, channels, proto_dim, use_cross_transfer=True, use_proto_conditioning=True):
        super().__init__()
        hidden = max(channels, 128)
        self.use_cross_transfer = bool(use_cross_transfer)
        self.use_proto_conditioning = bool(use_proto_conditioning)

        self.proto_to_channel = (
            nn.Sequential(nn.Linear(proto_dim, channels), nn.GELU())
            if self.use_proto_conditioning else None
        )
        self.router = nn.Sequential(
            nn.Linear(channels * 2, hidden), nn.GELU(), nn.Linear(hidden, channels * 3)
        )
        g = 8 if channels % 8 == 0 else 1
        self.lesion_norm = nn.GroupNorm(g, channels)
        self.severity_norm = nn.GroupNorm(g, channels)

        if self.use_cross_transfer:
            self.id_to_lesion = nn.Conv2d(channels, channels, 1, bias=False)
            self.lesion_to_severity = nn.Conv2d(channels, channels, 1, bias=False)
            self.alpha_id_to_lesion = nn.Parameter(torch.tensor(-2.0))
            self.alpha_lesion_to_severity = nn.Parameter(torch.tensor(-2.0))
        else:
            self.id_to_lesion = None
            self.lesion_to_severity = None
            self.register_parameter('alpha_id_to_lesion', None)
            self.register_parameter('alpha_lesion_to_severity', None)

    def forward(self, feature, soft_proto):
        b, c, _, _ = feature.shape
        feat_context = F.adaptive_avg_pool2d(feature, 1).flatten(1)

        if self.use_proto_conditioning:
            proto_context = self.proto_to_channel(soft_proto)
        else:
            # A03: remove prototype conditioning rather than replacing it with a learned static bias.
            proto_context = torch.zeros_like(feat_context)

        route_logits = self.router(torch.cat([feat_context, proto_context], 1)).view(b, 3, c, 1, 1)
        route_weights = F.softmax(route_logits, dim=1)

        identity_route = feature * route_weights[:, 0]
        lesion_base = feature * route_weights[:, 1]
        severity_base = feature * route_weights[:, 2]

        if self.use_cross_transfer:
            proto_gate = torch.sigmoid(proto_context)[:, :, None, None] if self.use_proto_conditioning else 1.0
            alpha = torch.sigmoid(self.alpha_id_to_lesion)
            beta = torch.sigmoid(self.alpha_lesion_to_severity)
            lesion_route = self.lesion_norm(
                feature + lesion_base + alpha * self.id_to_lesion(identity_route) * proto_gate
            )
            severity_route = self.severity_norm(
                feature + severity_base + beta * self.lesion_to_severity(lesion_route)
            )
        else:
            # A02: keep router/task partitioning, remove only directed cross-task transfer.
            lesion_route = self.lesion_norm(feature + lesion_base)
            severity_route = self.severity_norm(feature + severity_base)

        return identity_route, lesion_route, severity_route, route_weights

class AblationPhytoFactorHPCTRv2(nn.Module):
    def __init__(self, spec):
        super().__init__()
        self.spec = copy.deepcopy(spec)

        if cfg.pretrained:
            try:
                base = convnext_tiny(weights=ConvNeXt_Tiny_Weights.IMAGENET1K_V1)
            except Exception as e:
                raise RuntimeError('Enable Kaggle Internet so ImageNet weights can download. ' + str(e))
        else:
            base = convnext_tiny(weights=None)

        self.backbone = base.features
        self.stage_ids = {1, 3, 5, 7}
        channels = [96, 192, 384, 768]

        self.register_buffer('combo_to_host', COMBO_TO_HOST.clone(), persistent=False)
        self.register_buffer('combo_to_disease', COMBO_TO_DISEASE.clone(), persistent=False)

        self.embed = nn.Sequential(
            nn.LayerNorm(768), nn.Linear(768, cfg.embed_dim), nn.GELU(), nn.Dropout(cfg.dropout)
        )
        self.combo_head = nn.Linear(cfg.embed_dim, 115)
        self.host_head = nn.Linear(cfg.embed_dim, 34)
        self.disease_head = nn.Linear(cfg.embed_dim, 69)

        if self.spec['use_taxonomy_prior']:
            self.taxonomy_gate_logit = nn.Parameter(torch.tensor(-1.5))
        else:
            self.register_parameter('taxonomy_gate_logit', None)

        if self.spec['use_prototypes']:
            self.prototypes = nn.Parameter(torch.randn(115, cfg.embed_dim) * .02)
            self.proto_scale = nn.Parameter(torch.tensor(math.log(10.0)))
        else:
            self.register_parameter('prototypes', None)
            self.register_parameter('proto_scale', None)

        self.lateral = nn.ModuleList([nn.Conv2d(c, cfg.fpn_dim, 1) for c in channels])

        if self.spec['use_hpctr']:
            self.hpctr = nn.ModuleList([
                HPCTRBlock(
                    cfg.fpn_dim, cfg.embed_dim,
                    use_cross_transfer=self.spec['use_cross_transfer'],
                    use_proto_conditioning=self.spec['use_prototypes']
                ) for _ in range(4)
            ])
        else:
            self.hpctr = nn.ModuleList([SharedPassBlock(cfg.fpn_dim) for _ in range(4)])

        if self.spec['use_identity_refine']:
            self.identity_refine = nn.Sequential(
                nn.Linear(cfg.fpn_dim, cfg.embed_dim), nn.GELU(), nn.Dropout(cfg.dropout),
                nn.Linear(cfg.embed_dim, 115)
            )
        else:
            self.identity_refine = None

        self.smooth = nn.ModuleList([ConvGNAct(cfg.fpn_dim, cfg.fpn_dim) for _ in range(4)])
        self.seg_head = nn.Sequential(
            ConvGNAct(cfg.fpn_dim, 96), ConvGNAct(96, 64), nn.Conv2d(64, 1, 1)
        )

        self.severity_context = nn.Sequential(
            nn.Linear(cfg.fpn_dim * 4, cfg.embed_dim), nn.GELU(), nn.Dropout(cfg.dropout)
        )
        severity_in = cfg.embed_dim * 3 + 2
        self.direct_severity_head = nn.Sequential(
            nn.Linear(severity_in, 256), nn.GELU(), nn.Dropout(cfg.dropout),
            nn.Linear(256, 64), nn.GELU(), nn.Linear(64, 1)
        )

        if self.spec['severity_mode'] == 'uncertainty_gate':
            self.severity_gate_head = nn.Sequential(
                nn.Linear(severity_in, 128), nn.GELU(), nn.Linear(128, 1)
            )
            nn.init.constant_(self.severity_gate_head[-1].bias, 0.85)
        elif self.spec['severity_mode'] == 'fixed50':
            self.severity_gate_head = None
        else:
            raise ValueError(f"Unknown severity_mode: {self.spec['severity_mode']}")

    def features(self, x):
        outs = []
        for idx, layer in enumerate(self.backbone):
            x = layer(x)
            if idx in self.stage_ids:
                outs.append(x)
        return outs

    def forward(self, x):
        hw = x.shape[-2:]
        c1, c2, c3, c4 = self.features(x)
        z = self.embed(F.adaptive_avg_pool2d(c4, 1).flatten(1))

        host_logits = self.host_head(z)
        disease_logits = self.disease_head(z)

        # Prototype mechanism
        if self.spec['use_prototypes']:
            zn = F.normalize(z, dim=1)
            pn = F.normalize(self.prototypes, dim=1)
            proto_logits = (zn @ pn.t()) * self.proto_scale.exp().clamp(1, 50)
        else:
            pn = None
            proto_logits = z.new_zeros((z.size(0), 115))

        # Taxonomy prior
        if self.spec['use_taxonomy_prior']:
            host_logp = F.log_softmax(host_logits, dim=1)
            disease_logp = F.log_softmax(disease_logits, dim=1)
            taxonomy_prior = .5 * (
                host_logp[:, self.combo_to_host] + disease_logp[:, self.combo_to_disease]
            )
            taxonomy_gate = .35 * torch.sigmoid(self.taxonomy_gate_logit)
        else:
            taxonomy_prior = z.new_zeros((z.size(0), 115))
            taxonomy_gate = z.new_tensor(0.0)

        base_combo_logits = self.combo_head(z)
        if self.spec['use_prototypes']:
            base_combo_logits = base_combo_logits + .5 * proto_logits
        if self.spec['use_taxonomy_prior']:
            base_combo_logits = base_combo_logits + taxonomy_gate * taxonomy_prior

        if self.spec['use_prototypes']:
            soft_proto = F.softmax(base_combo_logits, dim=1) @ pn
        else:
            soft_proto = z.new_zeros((z.size(0), cfg.embed_dim))

        projected = [m(c) for m, c in zip(self.lateral, [c1, c2, c3, c4])]
        ids, lesions, severities, rws = [], [], [], []
        for feat, router in zip(projected, self.hpctr):
            rid, rles, rsev, rw = router(feat, soft_proto)
            ids.append(rid)
            lesions.append(rles)
            severities.append(rsev)
            rws.append(rw)

        combo_logits = base_combo_logits
        if self.identity_refine is not None:
            id_global = F.adaptive_avg_pool2d(ids[-1], 1).flatten(1)
            combo_logits = combo_logits + .25 * self.identity_refine(id_global)

        if self.spec['use_prototypes']:
            refined_soft_proto = F.softmax(combo_logits, dim=1) @ pn
        else:
            refined_soft_proto = z.new_zeros((z.size(0), cfg.embed_dim))

        # FPN-like lesion decoder from lesion route
        l1, l2, l3, l4 = lesions
        p4 = self.smooth[3](l4)
        p3 = self.smooth[2](l3 + F.interpolate(p4, size=l3.shape[-2:], mode='bilinear', align_corners=False))
        p2 = self.smooth[1](l2 + F.interpolate(p3, size=l2.shape[-2:], mode='bilinear', align_corners=False))
        p1 = self.smooth[0](l1 + F.interpolate(p2, size=l1.shape[-2:], mode='bilinear', align_corners=False))
        lesion_logits = F.interpolate(self.seg_head(p1), size=hw, mode='bilinear', align_corners=False)

        lesion_prob = torch.sigmoid(lesion_logits)
        coverage = lesion_prob.mean((1, 2, 3))
        uncertainty = (4 * lesion_prob * (1 - lesion_prob)).mean((1, 2, 3))

        sev_context = self.severity_context(torch.cat([
            F.adaptive_avg_pool2d(s, 1).flatten(1) for s in severities
        ], 1))

        uncertainty_feature = uncertainty.detach()[:, None]
        if self.spec['severity_mode'] == 'fixed50':
            # A06 removes use of uncertainty in the scalar fusion mechanism.
            uncertainty_feature = torch.zeros_like(uncertainty_feature)

        sev_input = torch.cat([
            z, refined_soft_proto, sev_context,
            coverage.detach()[:, None], uncertainty_feature
        ], 1)
        direct_severity = torch.sigmoid(self.direct_severity_head(sev_input).squeeze(1))

        if self.spec['severity_mode'] == 'uncertainty_gate':
            severity_gate = torch.sigmoid(self.severity_gate_head(sev_input).squeeze(1))
            severity = severity_gate * coverage.detach() + (1 - severity_gate) * direct_severity
        else:  # fixed50
            severity_gate = torch.full_like(coverage, 0.5)
            severity = 0.5 * coverage.detach() + 0.5 * direct_severity

        route_usage = torch.stack([rw.mean((0, 2, 3, 4)) for rw in rws]).mean(0)

        return dict(
            combo_logits=combo_logits,
            proto_logits=proto_logits,
            host_logits=host_logits,
            disease_logits=disease_logits,
            lesion_logits=lesion_logits,
            mask_coverage=coverage,
            mask_uncertainty=uncertainty,
            direct_severity=direct_severity,
            severity_gate=severity_gate,
            severity=severity,
            taxonomy_gate=taxonomy_gate,
            route_usage=route_usage
        )


In [9]:
# 8. Losses with ablation-aware switches
def dice_loss(logits, target, eps=1e-6):
    p = torch.sigmoid(logits)
    inter = (p * target).sum((1, 2, 3))
    den = p.sum((1, 2, 3)) + target.sum((1, 2, 3))
    return (1 - (2 * inter + eps) / (den + eps)).mean()

def focal_loss(logits, target, alpha=.75, gamma=2):
    bce = F.binary_cross_entropy_with_logits(logits, target, reduction='none')
    p = torch.sigmoid(logits)
    pt = p * target + (1 - p) * (1 - target)
    at = alpha * target + (1 - alpha) * (1 - target)
    return (at * (1 - pt).pow(gamma) * bce).mean()

def lovasz_grad(gt_sorted):
    p = len(gt_sorted)
    gts = gt_sorted.sum()
    intersection = gts - gt_sorted.float().cumsum(0)
    union = gts + (1 - gt_sorted.float()).cumsum(0)
    jaccard = 1 - intersection / union.clamp_min(1e-7)
    if p > 1:
        jaccard[1:p] = jaccard[1:p] - jaccard[:-1]
    return jaccard

def lovasz_hinge_flat(logits, labels):
    if logits.numel() == 0:
        return logits.sum() * 0
    labels = labels.float()
    signs = 2 * labels - 1
    errors = 1 - logits * signs
    errors_sorted, perm = torch.sort(errors, descending=True)
    gt_sorted = labels[perm.detach()]
    return torch.dot(F.relu(errors_sorted), lovasz_grad(gt_sorted))

def lovasz_hinge_loss(logits, target, downsample=256):
    logits_ds = F.interpolate(logits, size=(downsample, downsample), mode='bilinear', align_corners=False)
    target_ds = F.interpolate(target, size=(downsample, downsample), mode='nearest')
    return torch.stack([
        lovasz_hinge_flat(l.reshape(-1), t.reshape(-1)) for l, t in zip(logits_ds, target_ds)
    ]).mean()

def boundary_dice_loss(logits, target, k=3, eps=1e-6):
    p = torch.sigmoid(logits)
    pe = (F.max_pool2d(p, k, 1, k // 2) + F.max_pool2d(-p, k, 1, k // 2)).clamp(0, 1)
    te = (F.max_pool2d(target, k, 1, k // 2) + F.max_pool2d(-target, k, 1, k // 2)).clamp(0, 1)
    inter = (pe * te).sum((1, 2, 3))
    den = pe.sum((1, 2, 3)) + te.sum((1, 2, 3))
    return (1 - (2 * inter + eps) / (den + eps)).mean()

def js(p, q, eps=1e-7):
    p = p.clamp_min(eps)
    q = q.clamp_min(eps)
    m = .5 * (p + q)
    return .5 * (
        (p * (p.log() - m.log())).sum(1).mean() +
        (q * (q.log() - m.log())).sum(1).mean()
    )

C2H = COMBO_TO_HOST.to(DEVICE)
C2D = COMBO_TO_DISEASE.to(DEVICE)

def marginal(prob, mapping, n):
    out = torch.zeros(prob.size(0), n, device=prob.device, dtype=prob.dtype)
    out.scatter_add_(1, mapping[None].expand(prob.size(0), -1), prob)
    return out

def compute_loss(out, batch, spec, class_weights):
    cy = batch['combo'].to(DEVICE, non_blocking=True)
    hy = batch['host'].to(DEVICE, non_blocking=True)
    dy = batch['disease'].to(DEVICE, non_blocking=True)
    mask = batch['mask'].to(DEVICE, non_blocking=True)
    sy = batch['severity'].to(DEVICE, non_blocking=True)

    cw, hw, dw = class_weights
    if not spec['use_class_balancing']:
        cw = hw = dw = None

    combo_vec = F.cross_entropy(
        out['combo_logits'], cy, weight=cw, reduction='none', label_smoothing=cfg.label_smoothing
    )
    Lc = combo_vec.mean()
    Lh = F.cross_entropy(out['host_logits'], hy, weight=hw, label_smoothing=cfg.label_smoothing)
    Ld = F.cross_entropy(out['disease_logits'], dy, weight=dw, label_smoothing=cfg.label_smoothing)

    if spec['use_prototypes']:
        Lp = F.cross_entropy(out['proto_logits'], cy, weight=cw, label_smoothing=cfg.label_smoothing)
    else:
        Lp = Lc.new_zeros(())

    cp = F.softmax(out['combo_logits'], 1)
    hp = F.softmax(out['host_logits'], 1)
    dp = F.softmax(out['disease_logits'], 1)
    if spec['use_taxonomy_consistency']:
        Lt = js(hp, marginal(cp, C2H, 34)) + js(dp, marginal(cp, C2D, 69))
    else:
        Lt = Lc.new_zeros(())

    Ldice = dice_loss(out['lesion_logits'], mask)
    Lfocal = focal_loss(out['lesion_logits'], mask)

    if spec['seg_loss_mode'] == 'full':
        Llovasz = lovasz_hinge_loss(out['lesion_logits'], mask, cfg.lovasz_downsample)
        Lboundary = boundary_dice_loss(out['lesion_logits'], mask)
        Ls = .40 * Ldice + .30 * Lfocal + .20 * Llovasz + .10 * Lboundary
    elif spec['seg_loss_mode'] == 'dice_focal':
        Llovasz = Ldice.new_zeros(())
        Lboundary = Ldice.new_zeros(())
        Ls = .50 * Ldice + .50 * Lfocal
    else:
        raise ValueError(spec['seg_loss_mode'])

    if spec['use_factorized_loss']:
        bce_per = F.binary_cross_entropy_with_logits(
            out['lesion_logits'], mask, reduction='none'
        ).mean((1, 2, 3))
        lesion_frac = mask.mean((1, 2, 3)).detach()
        Lfactor = (bce_per + lesion_frac * combo_vec).mean()
    else:
        Lfactor = Lc.new_zeros(())

    Lsev_final = F.smooth_l1_loss(out['severity'], sy, beta=.05)
    Lsev_direct = F.smooth_l1_loss(out['direct_severity'], sy, beta=.05)
    Lsev = .80 * Lsev_final + .20 * Lsev_direct
    Lcons = (
        out['severity_gate'].detach() *
        torch.abs(out['severity'] - out['mask_coverage'].detach())
    ).mean()

    total = (
        cfg.w_combo * Lc +
        cfg.w_host * Lh +
        cfg.w_disease * Ld +
        cfg.w_taxonomy * Lt +
        cfg.w_prototype * Lp +
        cfg.w_seg * Ls +
        cfg.w_factorized * Lfactor +
        cfg.w_severity * Lsev +
        cfg.w_consistency * Lcons
    )

    parts = {
        'total': total.detach(), 'combo': Lc.detach(), 'host': Lh.detach(),
        'disease': Ld.detach(), 'taxonomy': Lt.detach(), 'prototype': Lp.detach(),
        'seg': Ls.detach(), 'dice': Ldice.detach(), 'focal': Lfocal.detach(),
        'lovasz': Llovasz.detach(), 'boundary': Lboundary.detach(),
        'factorized': Lfactor.detach(), 'severity': Lsev.detach(),
        'consistency': Lcons.detach()
    }
    return total, parts


In [10]:
# 9. Optimizer, EMA, evaluator, and checkpoint utilities
def build_param_groups(model):
    groups = {}
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if name.startswith('backbone.'):
            try:
                layer_idx = int(name.split('.')[1])
            except Exception:
                layer_idx = 7
            lr = cfg.backbone_lr * (cfg.layer_decay ** (7 - layer_idx))
        else:
            lr = cfg.head_lr

        no_decay = (
            p.ndim <= 1 or name.endswith('.bias') or 'prototypes' in name or
            'proto_scale' in name or 'gate_logit' in name or 'alpha_' in name
        )
        wd = 0.0 if no_decay else cfg.weight_decay
        key = (round(float(lr), 12), float(wd))
        groups.setdefault(key, {'params': [], 'lr': lr, 'weight_decay': wd})['params'].append(p)
    return list(groups.values())

class ModelEMA:
    def __init__(self, model, decay=.9995, warmup_tau=2000.):
        self.ema = copy.deepcopy(model).eval()
        self.decay = float(decay)
        self.warmup_tau = float(warmup_tau)
        self.updates = 0
        for p in self.ema.parameters():
            p.requires_grad_(False)

    @torch.no_grad()
    def update(self, model):
        self.updates += 1
        d = self.decay * (1 - math.exp(-self.updates / self.warmup_tau))
        es = self.ema.state_dict()
        ms = model.state_dict()
        for k, ev in es.items():
            mv = ms[k].detach()
            if torch.is_floating_point(ev):
                ev.mul_(d).add_(mv, alpha=1 - d)
            else:
                ev.copy_(mv)

@torch.no_grad()
def evaluate(model, loader, return_predictions=False, show_progress=False, desc='Evaluate'):
    model.eval()
    yt, yp, yht, yhp, ydt, ydp = [], [], [], [], [], []
    st, sp, files = [], [], []
    image_ious, image_dices = [], []
    conf = torch.zeros((116, 116), dtype=torch.int64, device=DEVICE)
    TP = FP = FN = 0.0

    iterator = (
        tqdm(loader, total=len(loader), desc=desc, dynamic_ncols=True, mininterval=.5, leave=False)
        if show_progress else loader
    )

    for batch in iterator:
        x = batch['image'].to(DEVICE, non_blocking=True)
        m = batch['mask'].to(DEVICE, non_blocking=True).bool()
        cy = batch['combo'].to(DEVICE, non_blocking=True)
        hy = batch['host'].to(DEVICE, non_blocking=True)
        dy = batch['disease'].to(DEVICE, non_blocking=True)

        with torch.amp.autocast(device_type=DEVICE.type, enabled=AMP_ENABLED):
            out = model(x)

        cp = out['combo_logits'].argmax(1)
        hp = out['host_logits'].argmax(1)
        dp = out['disease_logits'].argmax(1)
        pm = (torch.sigmoid(out['lesion_logits']) >= .5)

        tp = (pm & m).sum((1, 2, 3)).float()
        fp = (pm & ~m).sum((1, 2, 3)).float()
        fn = (~pm & m).sum((1, 2, 3)).float()
        piou = (tp + 1e-7) / (tp + fp + fn + 1e-7)
        pdice = (2 * tp + 1e-7) / (2 * tp + fp + fn + 1e-7)

        image_ious += piou.cpu().tolist()
        image_dices += pdice.cpu().tolist()
        TP += tp.sum().item(); FP += fp.sum().item(); FN += fn.sum().item()

        gs = m.squeeze(1).long() * (cy[:, None, None] + 1)
        ps = pm.squeeze(1).long() * (cp[:, None, None] + 1)
        flat = gs.reshape(-1) * 116 + ps.reshape(-1)
        conf += torch.bincount(flat, minlength=116 * 116).reshape(116, 116)

        yt += cy.cpu().tolist(); yp += cp.cpu().tolist()
        yht += hy.cpu().tolist(); yhp += hp.cpu().tolist()
        ydt += dy.cpu().tolist(); ydp += dp.cpu().tolist()
        st += batch['severity'].cpu().tolist()
        sp += out['severity'].float().cpu().tolist()
        files += batch['filename']

    c = conf.cpu().numpy().astype(float)
    inter = np.diag(c)
    union = c.sum(1) + c.sum(0) - inter
    support = c.sum(1)
    present = support[1:] > 0
    miou = np.mean(inter[1:][present] / np.maximum(union[1:][present], 1))
    macc = np.mean(inter[1:][present] / np.maximum(support[1:][present], 1))

    labs = sorted(set(yt)); hlabs = sorted(set(yht)); dlabs = sorted(set(ydt))
    metrics = dict(
        combo_accuracy=accuracy_score(yt, yp),
        combo_macro_f1=f1_score(yt, yp, labels=labs, average='macro', zero_division=0),
        combo_macro_precision=precision_score(yt, yp, labels=labs, average='macro', zero_division=0),
        combo_macro_recall=recall_score(yt, yp, labels=labs, average='macro', zero_division=0),
        host_accuracy=accuracy_score(yht, yhp),
        host_macro_f1=f1_score(yht, yhp, labels=hlabs, average='macro', zero_division=0),
        disease_accuracy=accuracy_score(ydt, ydp),
        disease_macro_f1=f1_score(ydt, ydp, labels=dlabs, average='macro', zero_division=0),
        binary_iou=TP / max(TP + FP + FN, 1),
        binary_dice=2 * TP / max(2 * TP + FP + FN, 1),
        binary_precision=TP / max(TP + FP, 1),
        binary_recall=TP / max(TP + FN, 1),
        binary_iou_image_macro=float(np.mean(image_ious)),
        binary_dice_image_macro=float(np.mean(image_dices)),
        semantic_miou_disease=float(miou),
        semantic_macc_disease=float(macc),
        severity_mae=mean_absolute_error(st, sp),
        severity_rmse=math.sqrt(mean_squared_error(st, sp)),
        severity_r2=r2_score(st, sp),
        severity_pearson=pearsonr(st, sp)[0] if np.std(st) > 0 and np.std(sp) > 0 else np.nan,
        severity_spearman=spearmanr(st, sp)[0] if np.std(st) > 0 and np.std(sp) > 0 else np.nan,
    )

    if not return_predictions:
        return metrics

    pred_df = pd.DataFrame({
        'filename': files,
        'true_combo': yt,
        'pred_combo': yp,
        'true_name': [COMBO_NAMES[i] for i in yt],
        'pred_name': [COMBO_NAMES[i] for i in yp],
        'true_severity': st,
        'pred_severity': sp,
        'image_binary_iou': image_ious,
        'image_binary_dice': image_dices,
    })
    return metrics, pred_df, c

def composite_score(m):
    # Exactly the main notebook's validation-selection score.
    return (
        m['semantic_miou_disease'] +
        .28 * m['combo_macro_f1'] +
        .10 * m['binary_dice'] +
        .03 * max(m['severity_r2'], 0) -
        .18 * m['severity_mae']
    )

def capture_rng_state():
    state = {
        'python': random.getstate(),
        'numpy': np.random.get_state(),
        'torch': torch.get_rng_state(),
    }
    if torch.cuda.is_available():
        state['cuda'] = torch.cuda.get_rng_state_all()
    return state

def restore_rng_state(state):
    if not state:
        return
    random.setstate(state['python'])
    np.random.set_state(state['numpy'])
    torch.set_rng_state(state['torch'])
    if torch.cuda.is_available() and 'cuda' in state:
        torch.cuda.set_rng_state_all(state['cuda'])


In [11]:
# 10. Independent training + test runner
# Shared settings used automatically by every dedicated ablation cell below.
DEFAULT_SEED = 42
SAVE_PREDICTIONS = True
RESUME = True
FORCE_RETRAIN = False

def run_ablation(ablation_key, run_seed=42, resume=True, force_retrain=False):
    if ablation_key not in ABLATIONS:
        raise KeyError(f'Unknown ablation: {ablation_key}')

    spec = copy.deepcopy(ABLATIONS[ablation_key])
    run_name = f'{ablation_key}__seed{int(run_seed)}'
    run_dir = Path(cfg.output_root) / run_name
    run_dir.mkdir(parents=True, exist_ok=True)

    result_csv = run_dir / 'test_result.csv'
    pred_csv = run_dir / 'test_predictions.csv'
    conf_npy = run_dir / 'semantic_confusion_matrix.npy'
    history_csv = run_dir / 'training_history.csv'
    best_path = run_dir / 'best_ema.pt'
    last_path = run_dir / 'last_checkpoint.pt'
    manifest_json = run_dir / 'run_manifest.json'

    if result_csv.exists() and not force_retrain:
        print(f'Already completed: {run_name}')
        print(f'Result: {result_csv}')
        return pd.read_csv(result_csv).iloc[0].to_dict()

    if force_retrain:
        for p in [result_csv, pred_csv, conf_npy, history_csv, best_path, last_path, manifest_json]:
            if p.exists():
                p.unlink()

    print('=' * 100)
    print('ABLATION:', ablation_key)
    print('GROUP   :', spec['group'])
    print('CHANGE  :', spec['change'])
    print('SEED    :', run_seed)
    print('=' * 100)

    seed_everything(run_seed)
    train_loader, val_loader, test_loader = make_loaders(run_seed)

    combo_w = combo_w_cpu.to(DEVICE)
    host_w = host_w_cpu.to(DEVICE)
    disease_w = disease_w_cpu.to(DEVICE)
    class_weights = (combo_w, host_w, disease_w)

    model = AblationPhytoFactorHPCTRv2(spec).to(DEVICE)
    params_m = sum(p.numel() for p in model.parameters()) / 1e6

    param_groups = build_param_groups(model)
    optimizer = torch.optim.AdamW(param_groups, betas=(.9, .999))
    steps_per_epoch = math.ceil(len(train_loader) / cfg.grad_accum)
    total_steps = cfg.epochs * steps_per_epoch
    warmup = cfg.warmup_epochs * steps_per_epoch

    def lr_factor(step):
        if step < warmup:
            return max(.05, (step + 1) / max(1, warmup))
        p = (step - warmup) / max(1, total_steps - warmup)
        return cfg.min_lr_ratio + (1 - cfg.min_lr_ratio) * .5 * (1 + math.cos(math.pi * min(1, p)))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_factor)
    scaler = torch.amp.GradScaler('cuda', enabled=AMP_ENABLED)
    ema = ModelEMA(model, cfg.ema_decay, cfg.ema_warmup_tau)

    history = []
    best_score = -1e9
    bad = 0
    start_epoch = 1
    cumulative_minutes = 0.0

    # Resume only this exact ablation/seed.
    if resume and last_path.exists() and not force_retrain:
        ckpt = torch.load(last_path, map_location=DEVICE, weights_only=False)
        if ckpt.get('ablation_key') != ablation_key or int(ckpt.get('seed', -1)) != int(run_seed):
            raise RuntimeError('Checkpoint identity does not match requested ablation/seed.')
        model.load_state_dict(ckpt['model'], strict=True)
        ema.ema.load_state_dict(ckpt['ema_model'], strict=True)
        optimizer.load_state_dict(ckpt['optimizer'])
        scheduler.load_state_dict(ckpt['scheduler'])
        scaler.load_state_dict(ckpt['scaler'])
        ema.updates = int(ckpt.get('ema_updates', 0))
        history = ckpt.get('history', [])
        best_score = float(ckpt.get('best_score', -1e9))
        bad = int(ckpt.get('bad', 0))
        start_epoch = int(ckpt['epoch']) + 1
        cumulative_minutes = float(ckpt.get('cumulative_minutes', 0.0))
        restore_rng_state(ckpt.get('rng_state'))
        print(f'Resumed {run_name} from epoch {start_epoch - 1}.')

    # One-batch preflight before a fresh run.
    if start_epoch == 1:
        model.train()
        optimizer.zero_grad(set_to_none=True)
        pre_batch = next(iter(train_loader))
        pre_x = pre_batch['image'].to(DEVICE, non_blocking=True)
        with torch.amp.autocast(device_type=DEVICE.type, enabled=AMP_ENABLED):
            pre_out = model(pre_x)
            pre_loss, pre_parts = compute_loss(pre_out, pre_batch, spec, class_weights)
        if not torch.isfinite(pre_loss):
            raise RuntimeError(f'Non-finite preflight loss: {pre_loss.item()}')
        scaler.scale(pre_loss).backward()
        optimizer.zero_grad(set_to_none=True)
        print(f'Preflight passed | total loss={pre_loss.item():.4f} | params={params_m:.3f}M')
        del pre_batch, pre_x, pre_out, pre_loss
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    manifest = {
        'ablation_key': ablation_key,
        'seed': int(run_seed),
        'group': spec['group'],
        'change': spec['change'],
        'spec': spec,
        'cfg': asdict(cfg),
        'parameters_m': float(params_m),
        'selection_metric': 'same composite validation score as main notebook',
        'test_protocol': 'single-view EMA only; untouched test split',
    }
    with open(manifest_json, 'w') as f:
        json.dump(manifest, f, indent=2)

    # Train/validate. A resumed run that had already early-stopped goes directly to testing.
    epochs_to_run = [] if bad >= cfg.patience else range(start_epoch, cfg.epochs + 1)
    if bad >= cfg.patience:
        print('Checkpoint had already reached early stopping; skipping further training and proceeding to test.')

    for epoch in epochs_to_run:
        epoch_start = time.time()
        model.train()
        optimizer.zero_grad(set_to_none=True)
        run = defaultdict(float)
        seen = 0

        train_bar = tqdm(
            enumerate(train_loader, 1), total=len(train_loader),
            desc=f'{ablation_key} | E{epoch:02d}/{cfg.epochs}',
            dynamic_ncols=True, mininterval=.5, leave=True
        )

        for step, batch in train_bar:
            x = batch['image'].to(DEVICE, non_blocking=True)
            with torch.amp.autocast(device_type=DEVICE.type, enabled=AMP_ENABLED):
                out = model(x)
                loss, parts = compute_loss(out, batch, spec, class_weights)
                scaled_loss = loss / cfg.grad_accum

            if not torch.isfinite(loss):
                raise RuntimeError(f'Non-finite loss epoch={epoch} step={step}: {loss.item()}')

            scaler.scale(scaled_loss).backward()

            if step % cfg.grad_accum == 0 or step == len(train_loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()
                ema.update(model)

            bs = x.size(0)
            seen += bs
            for k, v in parts.items():
                run[k] += float(v.item()) * bs

            if step == 1 or step % cfg.progress_every == 0 or step == len(train_loader):
                train_bar.set_postfix({
                    'loss': f"{run['total'] / max(seen, 1):.3f}",
                    'cls': f"{run['combo'] / max(seen, 1):.3f}",
                    'seg': f"{run['seg'] / max(seen, 1):.3f}",
                    'sev': f"{run['severity'] / max(seen, 1):.3f}",
                    'lr': f"{max(g['lr'] for g in optimizer.param_groups):.1e}"
                })

        train_losses = {f'train_{k}': v / max(seen, 1) for k, v in run.items()}
        vm = evaluate(
            ema.ema, val_loader, show_progress=True,
            desc=f'Val EMA | {ablation_key} | E{epoch:02d}'
        )
        score = composite_score(vm)
        epoch_minutes = (time.time() - epoch_start) / 60.0
        cumulative_minutes += epoch_minutes

        row = {
            'epoch': epoch,
            **train_losses,
            **{f'val_{k}': v for k, v in vm.items()},
            'selection_score': score,
            'epoch_minutes': epoch_minutes,
            'cumulative_minutes': cumulative_minutes,
        }
        history.append(row)
        pd.DataFrame(history).to_csv(history_csv, index=False)

        improved = score > best_score
        if improved:
            best_score = score
            bad = 0
            torch.save({
                'model': ema.ema.state_dict(),
                'epoch': int(epoch),
                'val_metrics': {k: float(v) for k, v in vm.items()},
                'best_score': float(best_score),
                'ablation_key': ablation_key,
                'seed': int(run_seed),
                'spec': spec,
            }, best_path)
            status = 'NEW BEST'
        else:
            bad += 1
            status = f'no improve {bad}/{cfg.patience}'

        torch.save({
            'model': model.state_dict(),
            'ema_model': ema.ema.state_dict(),
            'optimizer': optimizer.state_dict(),
            'scheduler': scheduler.state_dict(),
            'scaler': scaler.state_dict(),
            'epoch': int(epoch),
            'history': history,
            'best_score': float(best_score),
            'bad': int(bad),
            'ema_updates': int(ema.updates),
            'cumulative_minutes': float(cumulative_minutes),
            'rng_state': capture_rng_state(),
            'ablation_key': ablation_key,
            'seed': int(run_seed),
            'spec': spec,
        }, last_path)

        print(
            f"Epoch {epoch:02d} | {status} | "
            f"val F1={vm['combo_macro_f1']:.4f} | "
            f"sem mIoU={vm['semantic_miou_disease']:.4f} | "
            f"Dice={vm['binary_dice']:.4f} | "
            f"MAE={vm['severity_mae']:.4f} | R2={vm['severity_r2']:.4f}"
        )

        if bad >= cfg.patience:
            print(f'Early stopping at epoch {epoch}.')
            break

    if not best_path.exists():
        raise RuntimeError('Best EMA checkpoint is missing; training did not produce a valid checkpoint.')

    # Final untouched test evaluation — exactly once per completed training run.
    best = torch.load(best_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(best['model'], strict=True)
    model.eval()

    print(f"Testing best EMA checkpoint from epoch {best['epoch']}...")
    test_metrics, pred_df, semantic_conf = evaluate(
        model, test_loader, return_predictions=True, show_progress=True,
        desc=f'Final TEST | {ablation_key} | seed {run_seed}'
    )

    result_row = {
        'ablation_key': ablation_key,
        'group': spec['group'],
        'change': spec['change'],
        'seed': int(run_seed),
        'best_epoch': int(best['epoch']),
        'best_val_score': float(best['best_score']),
        'parameters_m': float(params_m),
        'training_minutes': float(cumulative_minutes),
        **{k: float(v) for k, v in test_metrics.items()},
    }

    # One-row CSV required for later aggregation.
    pd.DataFrame([result_row]).to_csv(result_csv, index=False)

    # Automatically refresh a root-level master CSV after every completed ablation.
    completed_files = sorted(Path(cfg.output_root).glob('A*__seed*/test_result.csv'))
    if completed_files:
        live_master = pd.concat([pd.read_csv(p) for p in completed_files], ignore_index=True)
        live_master = live_master.sort_values(['group', 'ablation_key', 'seed']).reset_index(drop=True)
        live_master.to_csv(Path(cfg.output_root) / 'ablation_runs_live.csv', index=False)

    if SAVE_PREDICTIONS:
        pred_df.to_csv(pred_csv, index=False)
        np.save(conf_npy, semantic_conf)

    print('\nCOMPLETED:', run_name)
    print('Result CSV :', result_csv)
    print('Master CSV :', Path(cfg.output_root) / 'ablation_runs_live.csv')
    if SAVE_PREDICTIONS:
        print('Predictions:', pred_csv)
    display(pd.DataFrame([result_row]))

    # Release GPU memory so another ablation can be run later in the same notebook session.
    del model, ema, optimizer, scheduler, scaler, train_loader, val_loader, test_loader
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return result_row


In [14]:
# A01 — Remove HPCTR routing
# Run this cell by itself for this ablation, or let Run All execute it in sequence.
A01_result = run_ablation(
    'A01_no_hpctr_routing',
    run_seed=DEFAULT_SEED,
    resume=RESUME,
    force_retrain=FORCE_RETRAIN,
)


ABLATION: A01_no_hpctr_routing
GROUP   : core_architecture
CHANGE  : HPCTR router removed; all tasks receive the same projected feature at each scale.
SEED    : 42
Downloading: "https://download.pytorch.org/models/convnext_tiny-983f1562.pth" to /root/.cache/torch/hub/checkpoints/convnext_tiny-983f1562.pth


100%|██████████| 109M/109M [00:00<00:00, 217MB/s]  


Preflight passed | total loss=6.6076 | params=29.553M


A01_no_hpctr_routing | E01/34:   0%|          | 0/3958 [00:00<?, ?it/s]

Val EMA | A01_no_hpctr_routing | E01:   0%|          | 0/624 [00:00<?, ?it/s]

...

Val EMA | A01_no_hpctr_routing | E28:   0%|          | 0/624 [00:00<?, ?it/s]

Epoch 28 | no improve 9/9 | val F1=0.7386 | sem mIoU=0.4337 | Dice=0.8168 | MAE=0.0536 | R2=0.7506
Early stopping at epoch 28.
Testing best EMA checkpoint from epoch 19...


Final TEST | A01_no_hpctr_routing | seed 42:   0%|          | 0/1148 [00:00<?, ?it/s]


COMPLETED: A01_no_hpctr_routing__seed42
Result CSV : /kaggle/working/ablation/A01_no_hpctr_routing__seed42/test_result.csv
Master CSV : /kaggle/working/ablation/ablation_runs_live.csv
Predictions: /kaggle/working/ablation/A01_no_hpctr_routing__seed42/test_predictions.csv


,ablation_key,group,change,seed,best_epoch,best_val_score,parameters_m,training_minutes,combo_accuracy,combo_macro_f1,...,binary_recall,binary_iou_image_macro,binary_dice_image_macro,semantic_miou_disease,semantic_macc_disease,severity_mae,severity_rmse,severity_r2,severity_pearson,severity_spearman
0,A01_no_hpctr_routing,core_architecture,HPCTR router removed; all tasks receive the sa...,42,19,0.746845,29.553074,194.804542,0.816122,0.749164,...,0.835224,0.655292,0.762808,0.464445,0.628631,0.05202,0.095307,0.789111,0.889437,0.889551


In [15]:
# A02 — Remove cross-route transfer
# Run this cell by itself for this ablation, or let Run All execute it in sequence.
A02_result = run_ablation(
    'A02_no_cross_route_transfer',
    run_seed=DEFAULT_SEED,
    resume=RESUME,
    force_retrain=FORCE_RETRAIN,
)


ABLATION: A02_no_cross_route_transfer
GROUP   : core_architecture
CHANGE  : HPCTR routing retained, but identity→lesion and lesion→severity transfers are removed.
SEED    : 42
Preflight passed | total loss=6.1888 | params=30.014M


A02_no_cross_route_transfer | E01/34:   0%|          | 0/3958 [00:00<?, ?it/s]

Val EMA | A02_no_cross_route_transfer | E01:   0%|          | 0/624 [00:00<?, ?it/s]

Epoch 01 | NEW BEST | val F1=0.3029 | sem mIoU=0.1355 | Dice=0.7168 | MAE=0.0934 | R2=0.5534


A02_no_cross_route_transfer | E02/34:   0%|          | 0/3958 [00:00<?, ?it/s]

...

Val EMA | A02_no_cross_route_transfer | E27:   0%|          | 0/624 [00:00<?, ?it/s]

Epoch 27 | no improve 9/9 | val F1=0.7351 | sem mIoU=0.4432 | Dice=0.8209 | MAE=0.0524 | R2=0.7570
Early stopping at epoch 27.
Testing best EMA checkpoint from epoch 18...


Final TEST | A02_no_cross_route_transfer | seed 42:   0%|          | 0/1148 [00:00<?, ?it/s]


COMPLETED: A02_no_cross_route_transfer__seed42
Result CSV : /kaggle/working/ablation/A02_no_cross_route_transfer__seed42/test_result.csv
Master CSV : /kaggle/working/ablation/ablation_runs_live.csv
Predictions: /kaggle/working/ablation/A02_no_cross_route_transfer__seed42/test_predictions.csv


,ablation_key,group,change,seed,best_epoch,best_val_score,parameters_m,training_minutes,combo_accuracy,combo_macro_f1,...,binary_recall,binary_iou_image_macro,binary_dice_image_macro,semantic_miou_disease,semantic_macc_disease,severity_mae,severity_rmse,severity_r2,severity_pearson,severity_spearman
0,A02_no_cross_route_transfer,core_architecture,"HPCTR routing retained, but identity→lesion an...",42,18,0.750133,30.014386,197.393634,0.816122,0.74775,...,0.839408,0.660056,0.766936,0.471973,0.630235,0.050764,0.094533,0.792521,0.891666,0.892556


In [16]:
# A03 — Remove prototype mechanism
# Run this cell by itself for this ablation, or let Run All execute it in sequence.
A03_result = run_ablation(
    'A03_no_prototype_mechanism',
    run_seed=DEFAULT_SEED,
    resume=RESUME,
    force_retrain=FORCE_RETRAIN,
)


ABLATION: A03_no_prototype_mechanism
GROUP   : core_architecture
CHANGE  : Prototype classifier, prototype logit contribution, prototype loss, and prototype conditioning are removed.
SEED    : 42
Preflight passed | total loss=5.2332 | params=29.984M


A03_no_prototype_mechanism | E01/34:   0%|          | 0/3958 [00:00<?, ?it/s]

Val EMA | A03_no_prototype_mechanism | E01:   0%|          | 0/624 [00:00<?, ?it/s]

Epoch 01 | NEW BEST | val F1=0.3030 | sem mIoU=0.1395 | Dice=0.7260 | MAE=0.0872 | R2=0.5943


A03_no_prototype_mechanism | E02/34:   0%|          | 0/3958 [00:00<?, ?it/s]

...

Val EMA | A03_no_prototype_mechanism | E34:   0%|          | 0/624 [00:00<?, ?it/s]

Epoch 34 | no improve 1/9 | val F1=0.7427 | sem mIoU=0.4535 | Dice=0.8168 | MAE=0.0541 | R2=0.7493
Testing best EMA checkpoint from epoch 33...


Final TEST | A03_no_prototype_mechanism | seed 42:   0%|          | 0/1148 [00:00<?, ?it/s]


COMPLETED: A03_no_prototype_mechanism__seed42
Result CSV : /kaggle/working/ablation/A03_no_prototype_mechanism__seed42/test_result.csv
Master CSV : /kaggle/working/ablation/ablation_runs_live.csv
Predictions: /kaggle/working/ablation/A03_no_prototype_mechanism__seed42/test_predictions.csv


,ablation_key,group,change,seed,best_epoch,best_val_score,parameters_m,training_minutes,combo_accuracy,combo_macro_f1,...,binary_recall,binary_iou_image_macro,binary_dice_image_macro,semantic_miou_disease,semantic_macc_disease,severity_mae,severity_rmse,severity_r2,severity_pearson,severity_spearman
0,A03_no_prototype_mechanism,core_architecture,"Prototype classifier, prototype logit contribu...",42,33,0.763257,29.984441,257.973864,0.818301,0.750735,...,0.829815,0.656827,0.763084,0.473003,0.621104,0.051219,0.096587,0.783407,0.887302,0.890216


In [ ]:
# A04 — Remove taxonomy prior
# Run this cell by itself for this ablation, or let Run All execute it in sequence.
A04_result = run_ablation(
    'A04_no_taxonomy_prior',
    run_seed=DEFAULT_SEED,
    resume=RESUME,
    force_retrain=FORCE_RETRAIN,
)


ABLATION: A04_no_taxonomy_prior
GROUP   : core_architecture
CHANGE  : Bounded host/disease taxonomy prior is removed from 115-class combination logits.
SEED    : 42
Downloading: "https://download.pytorch.org/models/convnext_tiny-983f1562.pth" to /root/.cache/torch/hub/checkpoints/convnext_tiny-983f1562.pth


100%|██████████| 109M/109M [00:00<00:00, 168MB/s]  


Preflight passed | total loss=6.3454 | params=30.145M


A04_no_taxonomy_prior | E01/34:   0%|          | 0/3958 [00:00<?, ?it/s]

Val EMA | A04_no_taxonomy_prior | E01:   0%|          | 0/624 [00:00<?, ?it/s]

...

Val EMA | A04_no_taxonomy_prior | E19:   0%|          | 0/624 [00:00<?, ?it/s]

Epoch 19 | no improve 9/9 | val F1=0.7388 | sem mIoU=0.4490 | Dice=0.8201 | MAE=0.0525 | R2=0.7593
Early stopping at epoch 19.
Testing best EMA checkpoint from epoch 10...


Final TEST | A04_no_taxonomy_prior | seed 42:   0%|          | 0/1148 [00:00<?, ?it/s]


COMPLETED: A04_no_taxonomy_prior__seed42
Result CSV : /kaggle/working/ablation/A04_no_taxonomy_prior__seed42/test_result.csv
Master CSV : /kaggle/working/ablation/ablation_runs_live.csv
Predictions: /kaggle/working/ablation/A04_no_taxonomy_prior__seed42/test_predictions.csv


,ablation_key,group,change,seed,best_epoch,best_val_score,parameters_m,training_minutes,combo_accuracy,combo_macro_f1,...,binary_recall,binary_iou_image_macro,binary_dice_image_macro,semantic_miou_disease,semantic_macc_disease,severity_mae,severity_rmse,severity_r2,severity_pearson,severity_spearman
0,A04_no_taxonomy_prior,core_architecture,Bounded host/disease taxonomy prior is removed...,42,10,0.75253,30.145465,148.628949,0.813508,0.750698,...,0.846789,0.65148,0.759674,0.468493,0.640206,0.052212,0.092772,0.800178,0.895192,0.890058


In [ ]:
# A05 — Remove identity refinement
# Run this cell by itself for this ablation, or let Run All execute it in sequence.
A05_result = run_ablation(
    'A05_no_identity_refinement',
    run_seed=DEFAULT_SEED,
    resume=RESUME,
    force_retrain=FORCE_RETRAIN,
)


ABLATION: A05_no_identity_refinement
GROUP   : core_architecture
CHANGE  : Final identity-route refinement of combination logits is removed.
SEED    : 42
Preflight passed | total loss=6.4703 | params=30.083M


A05_no_identity_refinement | E01/34:   0%|          | 0/3958 [00:00<?, ?it/s]

Val EMA | A05_no_identity_refinement | E01:   0%|          | 0/624 [00:00<?, ?it/s]

Epoch 01 | NEW BEST | val F1=0.3205 | sem mIoU=0.1420 | Dice=0.7168 | MAE=0.0933 | R2=0.5577


A05_no_identity_refinement | E02/34:   0%|          | 0/3958 [00:00<?, ?it/s]

...

Val EMA | A05_no_identity_refinement | E34:   0%|          | 0/624 [00:00<?, ?it/s]

Epoch 34 | no improve 9/9 | val F1=0.7295 | sem mIoU=0.4400 | Dice=0.8204 | MAE=0.0524 | R2=0.7587
Early stopping at epoch 34.
Testing best EMA checkpoint from epoch 25...


Final TEST | A05_no_identity_refinement | seed 42:   0%|          | 0/1148 [00:00<?, ?it/s]


COMPLETED: A05_no_identity_refinement__seed42
Result CSV : /kaggle/working/ablation/A05_no_identity_refinement__seed42/test_result.csv
Master CSV : /kaggle/working/ablation/ablation_runs_live.csv
Predictions: /kaggle/working/ablation/A05_no_identity_refinement__seed42/test_predictions.csv


,ablation_key,group,change,seed,best_epoch,best_val_score,parameters_m,training_minutes,combo_accuracy,combo_macro_f1,...,binary_recall,binary_iou_image_macro,binary_dice_image_macro,semantic_miou_disease,semantic_macc_disease,severity_mae,severity_rmse,severity_r2,severity_pearson,severity_spearman
0,A05_no_identity_refinement,core_architecture,Final identity-route refinement of combination...,42,25,0.753253,30.082887,262.627217,0.813943,0.743975,...,0.833729,0.659798,0.765707,0.472636,0.62417,0.050274,0.094796,0.791364,0.891422,0.893297


In [ ]:
# A06 — Remove uncertainty-aware severity gate
# Run this cell by itself for this ablation, or let Run All execute it in sequence.
A06_result = run_ablation(
    'A06_no_uncertainty_gate',
    run_seed=DEFAULT_SEED,
    resume=RESUME,
    force_retrain=FORCE_RETRAIN,
)


ABLATION: A06_no_uncertainty_gate
GROUP   : core_architecture
CHANGE  : Learned uncertainty-aware severity gate is replaced by fixed 50/50 mask/direct fusion; uncertainty input is zeroed.
SEED    : 42
Preflight passed | total loss=6.3432 | params=30.047M


A06_no_uncertainty_gate | E01/34:   0%|          | 0/3958 [00:00<?, ?it/s]

Val EMA | A06_no_uncertainty_gate | E01:   0%|          | 0/624 [00:00<?, ?it/s]

Epoch 01 | NEW BEST | val F1=0.2940 | sem mIoU=0.1328 | Dice=0.7270 | MAE=0.0908 | R2=0.5683


A06_no_uncertainty_gate | E02/34:   0%|          | 0/3958 [00:00<?, ?it/s]

...

Val EMA | A06_no_uncertainty_gate | E20:   0%|          | 0/624 [00:00<?, ?it/s]

Epoch 20 | no improve 9/9 | val F1=0.7370 | sem mIoU=0.4436 | Dice=0.8194 | MAE=0.0549 | R2=0.7642
Early stopping at epoch 20.
Testing best EMA checkpoint from epoch 11...


Final TEST | A06_no_uncertainty_gate | seed 42:   0%|          | 0/1148 [00:00<?, ?it/s]


COMPLETED: A06_no_uncertainty_gate__seed42
Result CSV : /kaggle/working/ablation/A06_no_uncertainty_gate__seed42/test_result.csv
Master CSV : /kaggle/working/ablation/ablation_runs_live.csv
Predictions: /kaggle/working/ablation/A06_no_uncertainty_gate__seed42/test_predictions.csv


,ablation_key,group,change,seed,best_epoch,best_val_score,parameters_m,training_minutes,combo_accuracy,combo_macro_f1,...,binary_recall,binary_iou_image_macro,binary_dice_image_macro,semantic_miou_disease,semantic_macc_disease,severity_mae,severity_rmse,severity_r2,severity_pearson,severity_spearman
0,A06_no_uncertainty_gate,core_architecture,Learned uncertainty-aware severity gate is rep...,42,11,0.749171,30.046649,155.564357,0.8122,0.745583,...,0.845462,0.653868,0.761622,0.466498,0.634179,0.054412,0.09456,0.792402,0.890479,0.886897


In [ ]:
# A07 — Remove factorized coupling loss
# Run this cell by itself for this ablation, or let Run All execute it in sequence.
A07_result = run_ablation(
    'A07_no_factorized_coupling',
    run_seed=DEFAULT_SEED,
    resume=RESUME,
    force_retrain=FORCE_RETRAIN,
)


ABLATION: A07_no_factorized_coupling
GROUP   : objective_support
CHANGE  : Lesion-weighted factorized classification/segmentation coupling loss is removed.
SEED    : 42
Preflight passed | total loss=5.9733 | params=30.145M


A07_no_factorized_coupling | E01/34:   0%|          | 0/3958 [00:00<?, ?it/s]

Val EMA | A07_no_factorized_coupling | E01:   0%|          | 0/624 [00:00<?, ?it/s]

Epoch 01 | NEW BEST | val F1=0.3282 | sem mIoU=0.1334 | Dice=0.7159 | MAE=0.0921 | R2=0.5495


A07_no_factorized_coupling | E02/34:   0%|          | 0/3958 [00:00<?, ?it/s]

...

Val EMA | A07_no_factorized_coupling | E20:   0%|          | 0/624 [00:00<?, ?it/s]

Epoch 20 | no improve 9/9 | val F1=0.7308 | sem mIoU=0.4340 | Dice=0.8183 | MAE=0.0531 | R2=0.7548
Early stopping at epoch 20.
Testing best EMA checkpoint from epoch 11...


Final TEST | A07_no_factorized_coupling | seed 42:   0%|          | 0/1148 [00:00<?, ?it/s]


COMPLETED: A07_no_factorized_coupling__seed42
Result CSV : ablation\A07_no_factorized_coupling__seed42\test_result.csv
Master CSV : ablation\ablation_runs_live.csv
Predictions: ablation\A07_no_factorized_coupling__seed42\test_predictions.csv


,ablation_key,group,change,seed,best_epoch,best_val_score,parameters_m,training_minutes,combo_accuracy,combo_macro_f1,...,binary_recall,binary_iou_image_macro,binary_dice_image_macro,semantic_miou_disease,semantic_macc_disease,severity_mae,severity_rmse,severity_r2,severity_pearson,severity_spearman
0,A07_no_factorized_coupling,objective_support,Lesion-weighted factorized classification/segm...,42,11,0.747754,30.145466,132.948713,0.809586,0.741647,...,0.855189,0.649317,0.756757,0.454153,0.633395,0.052961,0.095487,0.788313,0.88886,0.884098


In [ ]:
# A08 — Remove taxonomy consistency loss
# Run this cell by itself for this ablation, or let Run All execute it in sequence.
A08_result = run_ablation(
    'A08_no_taxonomy_consistency',
    run_seed=DEFAULT_SEED,
    resume=RESUME,
    force_retrain=FORCE_RETRAIN,
)


ABLATION: A08_no_taxonomy_consistency
GROUP   : objective_support
CHANGE  : JS consistency between host/disease heads and combination marginals is removed.
SEED    : 42
Preflight passed | total loss=6.3257 | params=30.145M


A08_no_taxonomy_consistency | E01/34:   0%|          | 0/3958 [00:00<?, ?it/s]

Val EMA | A08_no_taxonomy_consistency | E01:   0%|          | 0/624 [00:00<?, ?it/s]

Epoch 01 | NEW BEST | val F1=0.3377 | sem mIoU=0.1396 | Dice=0.7199 | MAE=0.0902 | R2=0.5640


A08_no_taxonomy_consistency | E02/34:   0%|          | 0/3958 [00:00<?, ?it/s]

...

Epoch 18 | no improve 2/9 | val F1=0.7388 | sem mIoU=0.4369 | Dice=0.8192 | MAE=0.0532 | R2=0.7616


A08_no_taxonomy_consistency | E19/34:   0%|          | 0/3958 [00:00<?, ?it/s]

Val EMA | A08_no_taxonomy_consistency | E19:   0%|          | 0/624 [00:00<?, ?it/s]

Epoch 19 | no improve 3/9 | val F1=0.7322 | sem mIoU=0.4323 | Dice=0.8187 | MAE=0.0532 | R2=0.7566


A08_no_taxonomy_consistency | E20/34:   0%|          | 0/3958 [00:00<?, ?it/s]

In [18]:

# ============================================================
# A08 — TEST ONLY FROM EXISTING BEST EMA CHECKPOINT
# NO TRAINING / NO RESUME
# ============================================================

from pathlib import Path
import torch
import pandas as pd
import numpy as np

ABLA = 'A08_no_taxonomy_consistency'
SEED = 42

root = Path(cfg.output_root)
run_dir = root / f'{ABLA}__seed{SEED}'

best_path = run_dir / 'best_ema.pt'
last_path = run_dir / 'last_checkpoint.pt'

# ------------------------------------------------------------
# 1. Check checkpoint
# ------------------------------------------------------------
if not best_path.exists():
    raise FileNotFoundError(
        f'\nA08 best_ema.pt not found:\n{best_path}'
    )

spec = ABLATIONS[ABLA]

# ------------------------------------------------------------
# 2. Load BEST EMA checkpoint
# ------------------------------------------------------------
best = torch.load(
    best_path,
    map_location=DEVICE,
    weights_only=False
)

print('=' * 80)
print('A08 TEST-ONLY EVALUATION')
print('=' * 80)
print(f'Checkpoint : {best_path}')
print(f'Best epoch : {best["epoch"]}')
print(f'Best score : {best["best_score"]:.6f}')
print('Training   : SKIPPED')
print('=' * 80)

# ------------------------------------------------------------
# 3. Build the exact model used in the notebook
#    IMPORTANT: do NOT use build_model()
# ------------------------------------------------------------
model = AblationPhytoFactorHPCTRv2(spec).to(DEVICE)

# Load EMA weights
model.load_state_dict(
    best['model'],
    strict=True
)

model.eval()

print('Model loaded successfully.')

# ------------------------------------------------------------
# 4. Create TEST loader
# ------------------------------------------------------------
_, _, test_loader = make_loaders(SEED)

print(f'Test samples: {len(test_loader.dataset)}')

# ------------------------------------------------------------
# 5. TEST ONLY
# ------------------------------------------------------------
test_metrics, pred_df, semantic_conf = evaluate(
    model,
    test_loader,
    return_predictions=True,
    show_progress=True,
    desc=f'Final TEST | {ABLA} | seed {SEED}'
)

# ------------------------------------------------------------
# 6. Parameter count
# ------------------------------------------------------------
params_m = (
    sum(p.numel() for p in model.parameters()) / 1e6
)

# ------------------------------------------------------------
# 7. Recover training time from last checkpoint
#    best_ema.pt itself does not store cumulative_minutes
# ------------------------------------------------------------
training_minutes = 0.0

if last_path.exists():
    last_ckpt = torch.load(
        last_path,
        map_location='cpu',
        weights_only=False
    )

    training_minutes = float(
        last_ckpt.get('cumulative_minutes', 0.0)
    )

# ------------------------------------------------------------
# 8. Create EXACT result schema used by run_ablation()
# ------------------------------------------------------------
result_row = {
    'ablation_key': ABLA,
    'group': spec['group'],
    'change': spec['change'],
    'seed': int(SEED),
    'best_epoch': int(best['epoch']),
    'best_val_score': float(best['best_score']),
    'parameters_m': float(params_m),
    'training_minutes': float(training_minutes),

    **{
        k: float(v)
        for k, v in test_metrics.items()
    }
}

# ------------------------------------------------------------
# 9. Save using EXACT filenames expected by notebook
# ------------------------------------------------------------
result_csv = run_dir / 'test_result.csv'
pred_csv = run_dir / 'test_predictions.csv'
conf_npy = run_dir / 'semantic_confusion_matrix.npy'

pd.DataFrame([result_row]).to_csv(
    result_csv,
    index=False
)

pred_df.to_csv(
    pred_csv,
    index=False
)

np.save(
    conf_npy,
    semantic_conf
)

# ------------------------------------------------------------
# 10. Refresh ablation_runs_live.csv
# ------------------------------------------------------------
result_files = sorted(
    root.glob('A*__seed*/test_result.csv')
)

if result_files:
    live_master = pd.concat(
        [
            pd.read_csv(p)
            for p in result_files
        ],
        ignore_index=True
    )

    live_master = (
        live_master
        .sort_values(
            ['group', 'ablation_key', 'seed']
        )
        .reset_index(drop=True)
    )

    live_csv = root / 'ablation_runs_live.csv'

    live_master.to_csv(
        live_csv,
        index=False
    )

else:
    live_master = pd.DataFrame()
    live_csv = root / 'ablation_runs_live.csv'

# ------------------------------------------------------------
# 11. Final confirmation
# ------------------------------------------------------------
print('\n')
print('=' * 80)
print('A08 RESULT SAVED SUCCESSFULLY')
print('=' * 80)

print(f'Result CSV    : {result_csv}')
print(f'Predictions   : {pred_csv}')
print(f'Confusion NPY : {conf_npy}')
print(f'Live master   : {live_csv}')
print(f'Master rows   : {len(live_master)}')

print('\nA08 TEST METRICS')
print('-' * 80)

display(
    pd.DataFrame([result_row])
)

A08 TEST-ONLY EVALUATION
Checkpoint : ablation\A08_no_taxonomy_consistency__seed42\best_ema.pt
Best epoch : 16
Best score : 0.753028
Training   : SKIPPED
Model loaded successfully.
Test samples: 2295


Final TEST | A08_no_taxonomy_consistency | seed 42:   0%|          | 0/1148 [00:00<?, ?it/s]



A08 RESULT SAVED SUCCESSFULLY
Result CSV    : ablation\A08_no_taxonomy_consistency__seed42\test_result.csv
Predictions   : ablation\A08_no_taxonomy_consistency__seed42\test_predictions.csv
Confusion NPY : ablation\A08_no_taxonomy_consistency__seed42\semantic_confusion_matrix.npy
Live master   : ablation\ablation_runs_live.csv
Master rows   : 10

A08 TEST METRICS
--------------------------------------------------------------------------------


,ablation_key,group,change,seed,best_epoch,best_val_score,parameters_m,training_minutes,combo_accuracy,combo_macro_f1,...,binary_recall,binary_iou_image_macro,binary_dice_image_macro,semantic_miou_disease,semantic_macc_disease,severity_mae,severity_rmse,severity_r2,severity_pearson,severity_spearman
0,A08_no_taxonomy_consistency,objective_support,JS consistency between host/disease heads and ...,42,16,0.753028,30.145466,128.366209,0.813943,0.747246,...,0.842988,0.658637,0.765094,0.472964,0.634134,0.051426,0.095306,0.789115,0.890121,0.888705


In [ ]:
# A09 — Use basic Dice + Focal segmentation loss
# Run this cell by itself for this ablation, or let Run All execute it in sequence.
A09_result = run_ablation(
    'A09_basic_segmentation_loss',
    run_seed=DEFAULT_SEED,
    resume=RESUME,
    force_retrain=FORCE_RETRAIN,
)


ABLATION: A09_basic_segmentation_loss
GROUP   : objective_support
CHANGE  : Segmentation objective is reduced to 0.5 Dice + 0.5 Focal; Lovasz and boundary terms are removed.
SEED    : 42
Preflight passed | total loss=5.8959 | params=30.145M


A09_basic_segmentation_loss | E01/34:   0%|          | 0/3958 [00:00<?, ?it/s]

Val EMA | A09_basic_segmentation_loss | E01:   0%|          | 0/624 [00:00<?, ?it/s]

Epoch 01 | NEW BEST | val F1=0.3365 | sem mIoU=0.1390 | Dice=0.7156 | MAE=0.0893 | R2=0.5738


A09_basic_segmentation_loss | E02/34:   0%|          | 0/3958 [00:00<?, ?it/s]

...

Val EMA | A09_basic_segmentation_loss | E24:   0%|          | 0/624 [00:00<?, ?it/s]

Epoch 24 | no improve 9/9 | val F1=0.7344 | sem mIoU=0.4384 | Dice=0.8211 | MAE=0.0522 | R2=0.7643
Early stopping at epoch 24.
Testing best EMA checkpoint from epoch 15...


Final TEST | A09_basic_segmentation_loss | seed 42:   0%|          | 0/1148 [00:00<?, ?it/s]


COMPLETED: A09_basic_segmentation_loss__seed42
Result CSV : ablation\A09_basic_segmentation_loss__seed42\test_result.csv
Master CSV : ablation\ablation_runs_live.csv
Predictions: ablation\A09_basic_segmentation_loss__seed42\test_predictions.csv


,ablation_key,group,change,seed,best_epoch,best_val_score,parameters_m,training_minutes,combo_accuracy,combo_macro_f1,...,binary_recall,binary_iou_image_macro,binary_dice_image_macro,semantic_miou_disease,semantic_macc_disease,severity_mae,severity_rmse,severity_r2,severity_pearson,severity_spearman
0,A09_basic_segmentation_loss,objective_support,Segmentation objective is reduced to 0.5 Dice ...,42,15,0.745192,30.145466,153.033118,0.815251,0.748804,...,0.844435,0.657336,0.76458,0.477741,0.639058,0.050829,0.093515,0.796965,0.895335,0.8924


In [ ]:
# A10 — Remove class balancing
# Run this cell by itself for this ablation, or let Run All execute it in sequence.
A10_result = run_ablation(
    'A10_no_class_balancing',
    run_seed=DEFAULT_SEED,
    resume=RESUME,
    force_retrain=FORCE_RETRAIN,
)


ABLATION: A10_no_class_balancing
GROUP   : objective_support
CHANGE  : Effective-number class weighting is removed from classification losses.
SEED    : 42
Preflight passed | total loss=8.8974 | params=30.145M


A10_no_class_balancing | E01/34:   0%|          | 0/3958 [00:00<?, ?it/s]

Val EMA | A10_no_class_balancing | E01:   0%|          | 0/624 [00:00<?, ?it/s]

Epoch 01 | NEW BEST | val F1=0.3020 | sem mIoU=0.1242 | Dice=0.7126 | MAE=0.0906 | R2=0.5476


A10_no_class_balancing | E02/34:   0%|          | 0/3958 [00:00<?, ?it/s]

...

Val EMA | A10_no_class_balancing | E30:   0%|          | 0/624 [00:00<?, ?it/s]

Epoch 30 | no improve 9/9 | val F1=0.7246 | sem mIoU=0.4370 | Dice=0.8194 | MAE=0.0520 | R2=0.7557
Early stopping at epoch 30.
Testing best EMA checkpoint from epoch 21...


Final TEST | A10_no_class_balancing | seed 42:   0%|          | 0/1148 [00:00<?, ?it/s]


COMPLETED: A10_no_class_balancing__seed42
Result CSV : ablation\A10_no_class_balancing__seed42\test_result.csv
Master CSV : ablation\ablation_runs_live.csv
Predictions: ablation\A10_no_class_balancing__seed42\test_predictions.csv


,ablation_key,group,change,seed,best_epoch,best_val_score,parameters_m,training_minutes,combo_accuracy,combo_macro_f1,...,binary_recall,binary_iou_image_macro,binary_dice_image_macro,semantic_miou_disease,semantic_macc_disease,severity_mae,severity_rmse,severity_r2,severity_pearson,severity_spearman
0,A10_no_class_balancing,objective_support,Effective-number class weighting is removed fr...,42,21,0.745702,30.145466,192.832264,0.815251,0.742096,...,0.835016,0.658518,0.764948,0.471573,0.619911,0.050773,0.095976,0.786139,0.888389,0.890272


In [19]:
# 11. Check completed/partial ablation runs without training anything
def scan_ablation_status():
    rows = []
    root = Path(cfg.output_root)
    for key in ABLATIONS:
        dirs = sorted(root.glob(f'{key}__seed*'))
        if not dirs:
            rows.append({'ablation_key': key, 'seed': np.nan, 'status': 'not_started', 'path': ''})
            continue
        for d in dirs:
            seed_match = re.search(r'__seed(\d+)$', d.name)
            seed = int(seed_match.group(1)) if seed_match else np.nan
            if (d / 'test_result.csv').exists():
                status = 'completed'
            elif (d / 'last_checkpoint.pt').exists():
                status = 'partial_resume_available'
            else:
                status = 'folder_only'
            rows.append({'ablation_key': key, 'seed': seed, 'status': status, 'path': str(d)})
    return pd.DataFrame(rows)

display(scan_ablation_status())


,ablation_key,seed,status,path
0,A01_no_hpctr_routing,42,completed,ablation\A01_no_hpctr_routing__seed42
1,A02_no_cross_route_transfer,42,completed,ablation\A02_no_cross_route_transfer__seed42
2,A03_no_prototype_mechanism,42,completed,ablation\A03_no_prototype_mechanism__seed42
3,A04_no_taxonomy_prior,42,completed,ablation\A04_no_taxonomy_prior__seed42
4,A05_no_identity_refinement,42,completed,ablation\A05_no_identity_refinement__seed42
5,A06_no_uncertainty_gate,42,completed,ablation\A06_no_uncertainty_gate__seed42
6,A07_no_factorized_coupling,42,completed,ablation\A07_no_factorized_coupling__seed42
7,A08_no_taxonomy_consistency,42,completed,ablation\A08_no_taxonomy_consistency__seed42
8,A09_basic_segmentation_loss,42,completed,ablation\A09_basic_segmentation_loss__seed42
9,A10_no_class_balancing,42,completed,ablation\A10_no_class_balancing__seed42


In [20]:
# 12. MERGE COMPLETED ABLATIONS — no training occurs here
MAIN_METRICS_JSON = None  # optional path to the main notebook's single-view metrics JSON
MAIN_PARAMETERS_M = None  # optional numeric value for the full proposed model

root = Path(cfg.output_root)
result_files = sorted(root.glob('A*__seed*/test_result.csv'))

if not result_files:
    print('No completed ablation result CSVs found yet.')
else:
    run_df = pd.concat([pd.read_csv(p) for p in result_files], ignore_index=True)
    run_df = run_df.sort_values(['group', 'ablation_key', 'seed']).reset_index(drop=True)
    run_df.to_csv(root / 'ablation_runs_all.csv', index=False)

    # Metrics emphasized in the paper-facing table.
    paper_metrics = [
        'combo_accuracy', 'combo_macro_f1',
        'binary_iou', 'binary_dice',
        'semantic_miou_disease', 'semantic_macc_disease',
        'severity_mae', 'severity_rmse', 'severity_r2'
    ]

    # Mean/std aggregation automatically works for 1 or multiple seeds.
    summary_rows = []
    for key, sub in run_df.groupby('ablation_key', sort=False):
        row = {
            'ablation_key': key,
            'group': sub['group'].iloc[0],
            'change': sub['change'].iloc[0],
            'n_seeds': int(sub['seed'].nunique()),
            'seeds': ','.join(map(str, sorted(sub['seed'].astype(int).unique().tolist()))),
            'parameters_m': float(sub['parameters_m'].mean()),
            'best_epoch_mean': float(sub['best_epoch'].mean()),
            'training_minutes_mean': float(sub['training_minutes'].mean()),
        }
        for m in paper_metrics:
            row[f'{m}_mean'] = float(sub[m].mean())
            row[f'{m}_std'] = float(sub[m].std(ddof=1)) if len(sub) > 1 else np.nan
        summary_rows.append(row)

    summary_df = pd.DataFrame(summary_rows)
    summary_df.to_csv(root / 'ablation_summary_mean_std.csv', index=False)

    # Compact publication table.
    compact = summary_df[[
        'ablation_key', 'group', 'n_seeds', 'parameters_m',
        'combo_accuracy_mean', 'combo_macro_f1_mean',
        'binary_dice_mean', 'semantic_miou_disease_mean',
        'severity_mae_mean', 'severity_r2_mean'
    ]].copy()

    # Optional proposed-model reference row from main notebook output.
    if MAIN_METRICS_JSON is not None and Path(MAIN_METRICS_JSON).exists():
        with open(MAIN_METRICS_JSON, 'r') as f:
            mm = json.load(f)
        main_row = {
            'ablation_key': 'Proposed_Full_from_main_notebook',
            'group': 'reference',
            'n_seeds': 1,
            'parameters_m': MAIN_PARAMETERS_M,
            'combo_accuracy_mean': mm.get('combo_accuracy', np.nan),
            'combo_macro_f1_mean': mm.get('combo_macro_f1', np.nan),
            'binary_dice_mean': mm.get('binary_dice', np.nan),
            'semantic_miou_disease_mean': mm.get('semantic_miou_disease', np.nan),
            'severity_mae_mean': mm.get('severity_mae', np.nan),
            'severity_r2_mean': mm.get('severity_r2', np.nan),
        }
        compact = pd.concat([pd.DataFrame([main_row]), compact], ignore_index=True)

        # Delta relative to the exact main single-view result.
        higher_better = [
            'combo_accuracy_mean', 'combo_macro_f1_mean', 'binary_dice_mean',
            'semantic_miou_disease_mean', 'severity_r2_mean'
        ]
        lower_better = ['severity_mae_mean']
        for m in higher_better:
            compact[f'delta_{m}'] = compact[m] - float(main_row[m])
        for m in lower_better:
            # Positive value means the ablation is worse than the proposed model.
            compact[f'delta_{m}'] = compact[m] - float(main_row[m])

    compact.to_csv(root / 'ablation_paper_table.csv', index=False)

    print('Merged files saved:')
    print(' -', root / 'ablation_runs_all.csv')
    print(' -', root / 'ablation_summary_mean_std.csv')
    print(' -', root / 'ablation_paper_table.csv')
    print('\nCompleted runs:', len(run_df))
    display(compact)


Merged files saved:
 - ablation\ablation_runs_all.csv
 - ablation\ablation_summary_mean_std.csv
 - ablation\ablation_paper_table.csv

Completed runs: 10


,ablation_key,group,n_seeds,parameters_m,combo_accuracy_mean,combo_macro_f1_mean,binary_dice_mean,semantic_miou_disease_mean,severity_mae_mean,severity_r2_mean
0,A01_no_hpctr_routing,core_architecture,1,29.553074,0.816122,0.749164,0.828041,0.464445,0.052020,0.789111
1,A02_no_cross_route_transfer,core_architecture,1,30.014386,0.816122,0.747750,0.830674,0.471973,0.050764,0.792521
2,A03_no_prototype_mechanism,core_architecture,1,29.984441,0.818301,0.750735,0.828366,0.473003,0.051219,0.783407
3,A04_no_taxonomy_prior,core_architecture,1,30.145465,0.813508,0.750698,0.825997,0.468493,0.052212,0.800178
4,A05_no_identity_refinement,core_architecture,1,30.082887,0.813943,0.743975,0.830491,0.472636,0.050274,0.791364
5,A06_no_uncertainty_gate,core_architecture,1,30.046649,0.812200,0.745583,0.827651,0.466498,0.054412,0.792402
6,A07_no_factorized_coupling,objective_support,1,30.145466,0.809586,0.741647,0.821772,0.454153,0.052961,0.788313
7,A08_no_taxonomy_consistency,objective_support,1,30.145466,0.813943,0.747246,0.829279,0.472964,0.051426,0.789115
8,A09_basic_segmentation_loss,objective_support,1,30.145466,0.815251,0.748804,0.831227,0.477741,0.050829,0.796965
9,A10_no_class_balancing,objective_support,1,30.145466,0.815251,0.742096,0.829409,0.471573,0.050773,0.786139


## Q1-quality reporting notes

For the manuscript, keep the **single-view EMA test protocol** as the primary ablation comparison, matching the main experiment. Do not select variants using test metrics; the notebook selects every checkpoint only from validation performance.

A strong final presentation is usually:

- **Table A — Core architecture ablation:** Proposed Full + A01–A06.
- **Table B — Objective/support ablation:** Proposed Full + A07–A10.
- Report Accuracy/Macro-F1 for 115-class recognition, Dice and semantic mIoU for localization/semantic quality, and Coverage MAE/R² for lesion coverage estimation.
- If compute permits, repeat the full model and the most important core ablations with multiple seeds and report mean ± standard deviation. This notebook already stores results by seed and aggregates them automatically.

The individual prediction CSVs are preserved so that a later paired bootstrap/significance analysis can be done without retraining.
